# Session 2 — What is in this table?
## 0 · How this notebook works
We have 640 emergency admissions from three hospitals and want to predict deterioration within 24 hours of arrival.
The data are synthetic


We meet the table, build a first model, then look at the data properly and rebuild.

Run cells in order. Every cell already works; do each **Task** before moving on.
If you fall behind, keep running: unfinished decisions remain unfinished, but the notebook still finishes.
Your numbers depend on the choices you enter. Restart and run from the top to reset the lesson.

## 1 · Setup and display options
Load the libraries and prepare a table for our results.
The display options show every column and keep printed tables readable.
Use a Python CPU runtime in Colab. The next cell installs the required packages before importing them.


In [ ]:
%pip install -q numpy pandas scikit-learn matplotlib


In [ ]:
from pathlib import Path
import hashlib
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score

SEED = 42
TARGET = "deterioration_24h"
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 60)
results = pd.DataFrame({"section": pd.Series(dtype="int64"),
                        "experiment": pd.Series(dtype="str"),
                        "setting": pd.Series(dtype="str"),
                        "auc": pd.Series(dtype="float64")})

def record(section, experiment, setting, auc):
    global results
    row = pd.DataFrame([[section, experiment, setting, float(auc)]], columns=results.columns)
    results = pd.concat([results, row], ignore_index=True)

plt.rcParams.update({"figure.dpi": 120, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Ready.")

In [ ]:
#@title Download the example CSV from GitHub (no edits needed)
CSV_URL = "https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/session-02-ed-deterioration/notebooks/session_02/data/ed_deterioration.csv"
CSV_SHA256 = "9d4fb82638032c046efc622d94da66ead80f67e7e38a93a268db43075a92212e"

DATA_DIR = Path("data_s2")
DATA_DIR.mkdir(exist_ok=True)
DATA_URL = DATA_DIR / "ed_deterioration.csv"
with urlopen(CSV_URL, timeout=60) as response:
    csv_bytes = response.read()
if hashlib.sha256(csv_bytes).hexdigest() != CSV_SHA256:
    raise ValueError("The downloaded CSV does not match this notebook's dataset version.")
DATA_URL.write_bytes(csv_bytes)
print("Data ready.")


**You should see:** Ready, followed by Data ready.

## 2 · Meet the table
Load the file and read its column names before doing anything to it.
Each row is a recorded admission; we will investigate what that means.

[Data dictionary: column descriptions and units](https://github.com/nbrg-ppcu/appliedmedtech/blob/session-02-ed-deterioration/notebooks/session_02/data/ed_deterioration_doc.md).


In [ ]:
ed = pd.read_csv(DATA_URL)
print("rows, columns:", ed.shape)
ed.head()

In [ ]:
print(ed.columns.tolist())
print(ed.dtypes)

**You should see:** the size of the table, five admissions, all column names and their stored types.

### Task 1
How many rows, and how many columns?

My answer: ___

### Task 2
Name three columns that describe the patient, and three that describe the hospital visit.

My answer: ___

## 3 · Find the target
The target is the column we are predicting; everything else starts as a candidate feature.
Never fit on the target. ROC-AUC compares how well predictions rank positive and negative cases: 0.5 is chance-level ranking, and 1 is perfect ranking.

In [ ]:
print(ed[TARGET].value_counts())
print(ed[TARGET].value_counts(normalize=True).round(3))

**You should see:** the count and fraction of each outcome.

### Task 3
What fraction of admissions deteriorated, and is the dataset balanced?

My answer: ___

## 4 · The obvious model, step by step
Turn the candidate columns into a matrix of numbers, then fit a first model.
Read the shapes after each step and write down the final score.

### 4.1 · Split the columns
Separate text from numbers and leave the target out.

In [ ]:
target = TARGET
features = [c for c in ed.columns if c != target]
text_cols = ed[features].select_dtypes(include=["object", "string"]).columns.tolist()
number_cols = [c for c in features if c not in text_cols]
print("candidate matrix:", ed[features].shape)
print("text columns:  ", text_cols)
print("number columns:", number_cols)

### 4.2 · One column per value
One-hot encoding replaces a text column with one new column per value; 3 values become 3 columns and 588 values become 588 columns.
Here we also create a column to represent a missing category. Count the expansion.

In [ ]:
X_text = pd.get_dummies(ed[text_cols], dummy_na=True)
print("before:", ed[text_cols].shape)
print("after: ", X_text.shape)
print("new columns created:", X_text.shape[1] - len(text_cols))
print(X_text.columns[:12].tolist())

### 4.3 · Fill the holes
Replace each missing number with the median of that column.

In [ ]:
X_num = ed[number_cols].fillna(ed[number_cols].median())
print("numeric matrix:", X_num.shape)
print("missing values left:", int(X_num.isna().sum().sum()))

### 4.4 · Join
Put the numeric and binary columns next to each other.

In [ ]:
X = pd.concat([X_num, X_text], axis=1)
y = ed[target]
print("final matrix:", X.shape)

### 4.5 · Split, fit, score
Train on one part and score predictions on the other part.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)
model = LogisticRegression(max_iter=3000)
model.fit(X_train, y_train)
obvious_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("training / test shapes:", X_train.shape, X_test.shape)
print(f"AUC: {obvious_auc:.3f}")
record(4, "obvious", "all candidate columns; first attempt", obvious_auc)

**You should see:** a large numeric matrix and one AUC.

### Task 4
Write your AUC down.

My answer: ___

### Task 5
Do you believe it? Write yes or no, and one reason.

My answer: ___

## 5 · Look properly
Run nine small checks and write down what each reveals.
Use the outputs as evidence for your decisions.

### 5.1 · Shape
How many rows support each column?

In [ ]:
print({"rows": len(ed), "columns": ed.shape[1], "rows_per_column": len(ed) / ed.shape[1]})

### 5.2 · Types
Which column holds numbers but is stored as text?

In [ ]:
print(ed.dtypes)
print("Text columns:", ed.select_dtypes(include=["object", "string"]).columns.tolist())

### Task 6
Which column holds numbers but is stored as text?

My answer: ___

### 5.3 · Missingness
Which three columns are emptiest?

In [ ]:
print((100 * ed.isna().mean()).sort_values(ascending=False).round(2).rename("percent_missing"))

### Task 7
Which three columns are emptiest?

My answer: ___

### 5.4 · Distinct values
Which text columns contain many different values?

In [ ]:
print(ed.select_dtypes(include=["object", "string"]).nunique(dropna=False).sort_values(ascending=False))

### 5.5 · Repeated identifiers
Is one row one patient?

In [ ]:
print(ed[["patient_id", "admission_id"]].apply(lambda column: column.duplicated().sum()).rename("repeated_rows"))

### Task 8
Is one row one patient? Use the two counts.

My answer: ___

### 5.6 · Constants
What information can a constant column carry?

In [ ]:
print(ed.columns[ed.nunique(dropna=False).eq(1)].tolist())

### 5.7 · Names
Which of these names describe the patient?

In [ ]:
print([c for c in ed.columns if any(part in c for part in ["_id", "date", "version"])])

### 5.8 · Derived values
Can one column be calculated from two others?

In [ ]:
computed_bmi = ed["weight_kg"] / (ed["height_cm"] / 100) ** 2
print(pd.DataFrame({"stored_bmi": ed["bmi"], "computed_bmi": computed_bmi, "absolute_difference": (ed["bmi"] - computed_bmi).abs()}).head(8).round(3))

### 5.9 · By hospital
Does any column change scale between hospitals?

In [ ]:
print(ed.groupby("site")[number_cols].median())

### Task 9
Does any column change scale between hospitals?

My answer: ___

**You should see:** nine sets of observations about the same file.
Read the data dictionary your instructor now provides. Compare its claims with what the file contains.

## 6 · Clean what is broken
Correct values and spellings before choosing how to represent them.
Work on a copy so that the original table remains available for comparison.

In [ ]:
ed_raw = pd.read_csv(DATA_URL)
ed = ed_raw.copy(deep=True)
print("Working copy:", ed.shape)

### 6.1 · A number stored as text
`temp_c` is body temperature in degrees Celsius; count values lost at each conversion.

In [ ]:
print("dtype:", ed["temp_c"].dtype)
print(ed["temp_c"].head(5).tolist())
naive = pd.to_numeric(ed_raw["temp_c"], errors="coerce")
print("naive conversion turns", int(naive.isna().sum()), "values into missing")
ed["temp_c"] = pd.to_numeric(ed_raw["temp_c"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
print("after fixing the comma:", int(ed["temp_c"].isna().sum()), "missing")

**You should see:** the number lost before and after the correction.
Always print how many values a conversion lost.

### 6.2 · One column, two scales
The generator uses a different creatinine unit at site C; convert it to the common scale.

In [ ]:
print("Before:")
print(ed_raw.groupby("site")["creatinine"].median().round(1))
ed["creatinine"] = ed_raw["creatinine"].where(ed_raw["site"] != "C", ed_raw["creatinine"] / 88.4)
print("After:")
print(ed.groupby("site")["creatinine"].median().round(2))

**You should see:** the hospital medians become closer after conversion.
Some repeated rows were generated by blending values in different units, so this correction does not repair every record.
Converting, dropping the column, or using a carefully checked hospital-specific representation needs a stated reason.

### 6.3 · Six spellings of two categories
Standardize the recorded spellings before counting categories.

In [ ]:
print("Before:")
print(ed_raw["sex"].value_counts())
ed["sex"] = ed_raw["sex"].str.upper().str[0]
print("After:")
print(ed["sex"].value_counts())

**You should see:** the original spellings combined into two recorded categories.

### Task 10
Using the dictionary, list every column you will not use and why, then fill DROP_COLS below.

My answer: ___

In [ ]:
DROP_COLS = [
    "",  # one column per line; add more lines as needed
]

In [ ]:
chosen_drop = [c for c in DROP_COLS if c and c in ed.columns and c != TARGET]
features = [c for c in ed.columns if c != TARGET and c not in chosen_drop]
text_cols = ed[features].select_dtypes(include=["object", "string"]).columns.tolist()
number_cols = [c for c in features if c not in text_cols]
X_text = pd.get_dummies(ed[text_cols], dummy_na=True)
X_num = ed[number_cols].fillna(ed[number_cols].median())
X = pd.concat([X_num, X_text], axis=1)
y = ed[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
model = LogisticRegression(max_iter=3000)
model.fit(X_train, y_train)
repair_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"With every column: {obvious_auc:.3f}     After your decisions: {repair_auc:.3f}")
if not chosen_drop:
    print("No column decisions recorded yet; this is still an unfinished first attempt.")
record(6, "after_decisions", "same first-attempt recipe on corrected/reduced columns", repair_auc)

**You should see:** the original score next to the score from your actual choices.
**Conclusion:** a score becomes defensible through justified inputs and evaluation, not through its size.
We still need to learn every fitted transformation from training rows alone.

## 7 · Columns into numbers, one at a time
Compare the same four admissions before and after each operation.
For a learned operation, fit on training rows and use that fitted object to transform other rows.

In [ ]:
DEMO_ROWS = [0, 1, 16, 21]
train_rows, test_rows = train_test_split(np.arange(len(ed)), test_size=0.25, random_state=SEED, stratify=ed[TARGET])
demo_train = ed.iloc[train_rows]

def show_change(before, after):
    print(pd.concat({"before": before.loc[DEMO_ROWS], "after": after.loc[DEMO_ROWS]}, axis=1).round(3))

print("Admissions used in every example:", DEMO_ROWS)

### 7.1 · One-hot
Represent site and recorded sex with one binary column per category.

In [ ]:
onehot_demo = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
onehot_demo.fit(demo_train[["site", "sex"]])
onehot_after = pd.DataFrame(onehot_demo.transform(ed[["site", "sex"]]),
                           columns=onehot_demo.get_feature_names_out(), index=ed.index)
show_change(ed[["site", "sex"]], onehot_after)
print("Output columns:", onehot_after.shape[1])
print("Additional columns:", onehot_after.shape[1] - 2)

**You should see:** the same four rows before and after.
One-hot encoding gives each unordered category its own binary column.
Integer category codes would suggest an order or distance that is not present.

### 7.2 · Indicator and imputation
Record whether lactate was missing, then fill its value. Change the flag here when you reach Task 12.

In [ ]:
ADD_INDICATOR = True
lactate_imputer = SimpleImputer(strategy="median")
lactate_imputer.fit(demo_train[["lactate"]])
lactate_after = pd.DataFrame(lactate_imputer.transform(ed[["lactate"]]), columns=["lactate"], index=ed.index)
if ADD_INDICATOR:
    lactate_after["lactate_missing"] = ed["lactate"].isna().astype(int)
show_change(ed[["lactate"]], lactate_after)
print("ADD_INDICATOR:", ADD_INDICATOR)

**You should see:** the same four rows before and after.
Keep a flag saying the measurement was missing, then fill its numeric value using the training median.
Filling alone removes the distinction between a measured median and an unmeasured value.

### 7.3 · A real order
Declare low, medium, high as the order of the Charlson bands.

In [ ]:
charlson_order = ["low", "medium", "high"]
ordinal_demo = OrdinalEncoder(categories=[charlson_order], handle_unknown="use_encoded_value", unknown_value=-1)
ordinal_demo.fit(demo_train[["charlson_band"]])
ordinal_after = pd.DataFrame(ordinal_demo.transform(ed[["charlson_band"]]), columns=["charlson_code"], index=ed.index)
show_change(ed[["charlson_band"]], ordinal_after)

**You should see:** the same four rows before and after.
Declare the low, medium, high order before encoding it as 0, 1, 2.
Alphabetical order would put high before low and misrepresent the declared order.

### 7.4 · Scaling
Put age and creatinine on a comparable numerical scale after filling missing values.

In [ ]:
scale_imputer = SimpleImputer(strategy="median")
scale_train = scale_imputer.fit_transform(demo_train[["age", "creatinine"]])
scale_all = scale_imputer.transform(ed[["age", "creatinine"]])
scaler_demo = StandardScaler()
scaler_demo.fit(scale_train)
scaled_after = pd.DataFrame(scaler_demo.transform(scale_all), columns=["age_scaled", "creatinine_scaled"], index=ed.index)
show_change(ed[["age", "creatinine"]], scaled_after)

**You should see:** the same four rows before and after.
Subtract the training mean and divide by the training standard deviation.
Fitting a second scaler on the test rows would give the two matrices different meanings.

### 7.5 · What an identifier costs
Count the columns that one-hot encoding patient_id would create; show only the four example patients below.

In [ ]:
patient_columns = pd.get_dummies(ed["patient_id"], dtype=int)
four_patient_columns = ed.loc[DEMO_ROWS, "patient_id"].unique().tolist()
show_change(ed[["patient_id"]], patient_columns[four_patient_columns])
print("All patient columns:", patient_columns.shape[1])
print("Additional columns compared with one ID column:", patient_columns.shape[1] - 1)
print("The table above shows only", len(four_patient_columns), "of those columns.")

**You should see:** the same four rows before and after.
Count what representing one category per patient would cost.
An identifier can encode identity without supplying a transferable patient characteristic.

### Task 11
Fill ONE_HOT_COLS and ORDINAL_COLS using the representations you chose.

My answer: ___

In [ ]:
ONE_HOT_COLS = [""]
ORDINAL_COLS = [""]

### 7.6 · Assemble by hand
Read each training fit and matching test transform. Then join the three groups in the same order.
Only the columns named in your lists receive text encodings; an empty list selects none.

In [ ]:
MODEL_COLUMNS = features.copy()
NOMINAL_COLS = [c for c in ONE_HOT_COLS if c in MODEL_COLUMNS]
ORDERED_COLS = [c for c in ORDINAL_COLS if c in MODEL_COLUMNS and c not in NOMINAL_COLS]
NUMERIC_COLS = [c for c in MODEL_COLUMNS if pd.api.types.is_numeric_dtype(ed[c]) and c not in NOMINAL_COLS + ORDERED_COLS]
ORDERS = {"charlson_band": ["low", "medium", "high"]}
ORDERED_COLS = [c for c in ORDERED_COLS if c in ORDERS]
train_frame, test_frame = ed.iloc[train_rows], ed.iloc[test_rows]
train_y, test_y = y.iloc[train_rows], y.iloc[test_rows]
print("Numeric:", NUMERIC_COLS)
print("One-hot:", NOMINAL_COLS, "Ordered:", ORDERED_COLS)

In [ ]:
train, test = train_frame, test_frame
add_indicator = ADD_INDICATOR

num_imputer = SimpleImputer(strategy="median", add_indicator=add_indicator)
train_num = num_imputer.fit_transform(train[NUMERIC_COLS])
test_num = num_imputer.transform(test[NUMERIC_COLS])
num_scaler = StandardScaler()
train_num = num_scaler.fit_transform(train_num)
test_num = num_scaler.transform(test_num)
train_nom, test_nom = np.empty((len(train), 0)), np.empty((len(test), 0))
nom_names = []
if NOMINAL_COLS:
    nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    train_nom = nominal.fit_transform(train[NOMINAL_COLS])
    test_nom = nominal.transform(test[NOMINAL_COLS])
    nom_names = nominal.get_feature_names_out().tolist()
train_ord, test_ord = np.empty((len(train), 0)), np.empty((len(test), 0))
if ORDERED_COLS:
    ordinal = OrdinalEncoder(categories=[ORDERS[c] for c in ORDERED_COLS], handle_unknown="use_encoded_value", unknown_value=-1)
    train_ord = ordinal.fit_transform(train[ORDERED_COLS])
    test_ord = ordinal.transform(test[ORDERED_COLS])
train_matrix = np.column_stack([train_num, train_nom, train_ord])
test_matrix = np.column_stack([test_num, test_nom, test_ord])
matrix_names = num_imputer.get_feature_names_out().tolist() + nom_names + ORDERED_COLS

manual_model = LogisticRegression(max_iter=3000)
manual_model.fit(train_matrix, train_y)
manual_auc = roc_auc_score(test_y, manual_model.predict_proba(test_matrix)[:, 1])
print("Train shape:", train_matrix.shape, "Test shape:", test_matrix.shape)
assembled = pd.concat([pd.DataFrame(train_matrix, columns=matrix_names, index=train.index),
                       pd.DataFrame(test_matrix, columns=matrix_names, index=test.index)])
show_change(ed[["age", "lactate", "site", "charlson_band"]], assembled)
print(f"Manual held-out AUC: {manual_auc:.3f}")
record(7, "manual_holdout", f"indicator={ADD_INDICATOR}", manual_auc)

**You should see:** the same four rows before and after.
Fit each object on training rows and transform test rows using that same object.
Separate preparation of the two halves can change column order, category mappings or scale.

Repeat those exact fitting steps inside each training fold to compare the indicator setting.
The model adds indicators for all numeric columns with missing values; the earlier lactate example showed just one.
The supplied helper below repeats the code you just read; no edits are needed.

In [ ]:
def manual_arrays(train, test, add_indicator=True):
    num_imputer = SimpleImputer(strategy="median", add_indicator=add_indicator)
    train_num = num_imputer.fit_transform(train[NUMERIC_COLS])
    test_num = num_imputer.transform(test[NUMERIC_COLS])
    num_scaler = StandardScaler()
    train_num = num_scaler.fit_transform(train_num)
    test_num = num_scaler.transform(test_num)
    train_nom, test_nom = np.empty((len(train), 0)), np.empty((len(test), 0))
    nom_names = []
    if NOMINAL_COLS:
        nominal = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        train_nom = nominal.fit_transform(train[NOMINAL_COLS])
        test_nom = nominal.transform(test[NOMINAL_COLS])
        nom_names = nominal.get_feature_names_out().tolist()
    train_ord, test_ord = np.empty((len(train), 0)), np.empty((len(test), 0))
    if ORDERED_COLS:
        ordinal = OrdinalEncoder(categories=[ORDERS[c] for c in ORDERED_COLS], handle_unknown="use_encoded_value", unknown_value=-1)
        train_ord = ordinal.fit_transform(train[ORDERED_COLS])
        test_ord = ordinal.transform(test[ORDERED_COLS])
    train_matrix = np.column_stack([train_num, train_nom, train_ord])
    test_matrix = np.column_stack([test_num, test_nom, test_ord])
    matrix_names = num_imputer.get_feature_names_out().tolist() + nom_names + ORDERED_COLS
    return train_matrix, test_matrix


def score_manual(frame, grouped=False, add_indicator=True):
    splitter = (StratifiedGroupKFold(5, shuffle=True, random_state=SEED) if grouped
                else StratifiedKFold(5, shuffle=True, random_state=SEED))
    groups = frame["patient_id"]
    folds = splitter.split(frame, frame[TARGET], groups) if grouped else splitter.split(frame, frame[TARGET])
    rows = []
    for fold, (train_index, test_index) in enumerate(folds, 1):
        train_part, test_part = frame.iloc[train_index], frame.iloc[test_index]
        a, b = manual_arrays(train_part, test_part, add_indicator)
        estimator = LogisticRegression(max_iter=3000)
        estimator.fit(a, train_part[TARGET])
        score = roc_auc_score(test_part[TARGET], estimator.predict_proba(b)[:, 1])
        overlap = set(train_part["patient_id"]) & set(test_part["patient_id"])
        rows.append({"fold": fold, "auc": score, "shared_patients": len(overlap)})
    return pd.DataFrame(rows)

indicator_on_folds = score_manual(ed, add_indicator=True)
indicator_off_folds = score_manual(ed, add_indicator=False)
indicator_on = indicator_on_folds["auc"].mean()
indicator_off = indicator_off_folds["auc"].mean()
print(f"Five-fold indicators on: {indicator_on:.3f}   off: {indicator_off:.3f}   gap: {indicator_on - indicator_off:+.3f}")
record(7, "indicator_comparison", "on; five-fold", indicator_on)
record(7, "indicator_comparison", "off; five-fold", indicator_off)

### Task 12
Run 7.2 with the indicator on, then off; rerun 7.6 and report the two five-fold scores and their gap.

My answer: ___

**Conclusion:** representing missingness changes this model’s score.
The fitted medians, scales and category mapping must come from training rows.

## 8 · One patient, two rows
Decide whether the test should represent a new admission or a new patient.
If the same patient occurs in both halves, familiar patient patterns can help predictions; measure the effect.

### 8.1 · How much repetition is there?
Count patients as well as admissions.

In [ ]:
counts = ed["patient_id"].value_counts()
print("admissions:", len(ed))
print("patients:  ", ed["patient_id"].nunique())
print(counts.value_counts().sort_index().rename("patients with this many admissions"))

### 8.2 · Two different questions
A row split can share patients between training and test. A grouped split keeps every row for a patient in one half.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3), layout="constrained")
row_assignment = np.array([[0, 1, 0, 0], [1, 0, 0, 1], [0, 0, 1, 0], [1, 0, 1, 0]])
group_assignment = np.array([[0, 0, 0, 0], [1, 1, 1, 1], [0, 0, 0, 0], [1, 1, 1, 1]])
for ax, values, title in zip(axes, [row_assignment, group_assignment], ["Split admissions", "Keep patients together"]):
    ax.imshow(values, cmap=plt.matplotlib.colors.ListedColormap(["#d9ecef", "#f0d2bb"]), vmin=0, vmax=1, aspect="auto")
    for row in range(4):
        for col in range(4):
            ax.text(col, row, "Test" if values[row, col] else "Train", ha="center", va="center", fontsize=10)
    ax.set(xticks=range(4), xticklabels=["Visit 1", "Visit 2", "Visit 3", "Visit 4"],
           yticks=range(4), yticklabels=["Patient A", "Patient B", "Patient C", "Patient D"], title=title)
plt.show()
print("Schematic only: four example patients, each with four visits.")

### 8.3 · Measure it here
Refit every transformation inside each fold and score ROC-AUC, keeping the same model settings.

In [ ]:
real_random_folds = score_manual(ed, grouped=False, add_indicator=ADD_INDICATOR)
real_grouped_folds = score_manual(ed, grouped=True, add_indicator=ADD_INDICATOR)
real_random = real_random_folds["auc"].mean()
real_grouped = real_grouped_folds["auc"].mean()
print(pd.DataFrame({"random_auc": real_random_folds["auc"], "grouped_auc": real_grouped_folds["auc"],
                    "random_shared_patients": real_random_folds["shared_patients"],
                    "grouped_shared_patients": real_grouped_folds["shared_patients"]}).round(3))
print(f"Random: {real_random:.3f}   Grouped: {real_grouped:.3f}   Random minus grouped: {real_random - real_grouped:+.3f}")
record(8, "original_patients", "random", real_random)
record(8, "original_patients", "grouped", real_grouped)

For Task 13, keep the model fixed and measure 15 train/test splits now.
Section 10 repeats this calculation after the steps have been assembled into one object.

In [ ]:
manual_seed_scores = []
for split_seed in range(15):
    a_rows, b_rows = train_test_split(np.arange(len(ed)), test_size=0.25,
                                      stratify=ed[TARGET], random_state=split_seed)
    a, b = manual_arrays(ed.iloc[a_rows], ed.iloc[b_rows], ADD_INDICATOR)
    estimator = LogisticRegression(max_iter=3000)
    estimator.fit(a, ed.iloc[a_rows][TARGET])
    manual_seed_scores.append(roc_auc_score(ed.iloc[b_rows][TARGET], estimator.predict_proba(b)[:, 1]))
split_range_manual = np.ptp(manual_seed_scores)
print(f"Range across 15 splits: {split_range_manual:.3f}")

**You should see:** two mean AUCs, counts of patients shared across folds, and the split-to-split range.

### Task 13
Is the random/grouped difference bigger or smaller than the measured split range, and what do you conclude?

My answer: ___

### 8.4 · When repetition is stronger
Make four measurements for each of 160 patients selected from this same table, with small changes to vitals and laboratory values.
We deliberately keep each patient’s baseline outcome unchanged; this is a controlled repeated-measurement example, not a realistic simulator of new admission outcomes.

In [ ]:
baseline_rows = ed.drop_duplicates("patient_id", keep="first").copy()
selected_rows, _ = train_test_split(np.arange(len(baseline_rows)), train_size=160,
                                  stratify=baseline_rows[TARGET], random_state=SEED)
base = baseline_rows.iloc[selected_rows].reset_index(drop=True)
followup = base.loc[base.index.repeat(4)].reset_index(drop=True)
visit = np.tile(np.arange(4), len(base))
rng = np.random.default_rng(SEED)
drifts = {"heart_rate": (3.0, 40, 175), "resp_rate": (0.8, 8, 46),
          "sbp": (4.0, 62, 205), "temp_c": (0.1, 34.5, 41.0), "spo2": (0.5, 74, 100),
          "wbc": (0.2, 0.8, 42), "creatinine": (0.03, 0.3, 6.5),
          "crp": (2.0, 0.4, 400), "lactate": (0.1, 0.3, 12)}
for column, (sd, lower, upper) in drifts.items():
    delta = rng.normal(0, sd, len(followup))
    delta[visit == 0] = 0
    followup[column] = (followup[column] + delta).clip(lower, upper)
followup["admission_id"] = followup["admission_id"] + "-V" + (visit + 1).astype(str)
followup["admit_date"] = (pd.to_datetime(followup["admit_date"]) + pd.to_timedelta(visit * 14, unit="D")).dt.strftime("%Y-%m-%d")
print("Follow-up rows:", len(followup), "Patients:", followup["patient_id"].nunique())
print(followup["patient_id"].value_counts().value_counts().rename("patients by visit count"))

In [ ]:
followup_random_folds = score_manual(followup, grouped=False, add_indicator=ADD_INDICATOR)
followup_grouped_folds = score_manual(followup, grouped=True, add_indicator=ADD_INDICATOR)
followup_random = followup_random_folds["auc"].mean()
followup_grouped = followup_grouped_folds["auc"].mean()
print(pd.DataFrame({"random_auc": followup_random_folds["auc"], "grouped_auc": followup_grouped_folds["auc"],
                    "random_shared_patients": followup_random_folds["shared_patients"],
                    "grouped_shared_patients": followup_grouped_folds["shared_patients"]}).round(3))
print(f"Random: {followup_random:.3f}   Grouped: {followup_grouped:.3f}   Random minus grouped: {followup_random - followup_grouped:+.3f}")
record(8, "followup_patients", "random", followup_random)
record(8, "followup_patients", "grouped", followup_grouped)

**You should see:** 160 patients with four measurements each, and the two measured scores.

### Task 14
What changed between 8.3 and 8.4?

My answer: ___

**Conclusion:** the intended use determines which rows must stay together; the experiment measures how much the score changes.
A small change does not make a row split a valid test of performance on new patients.
Amend the relevant Design Sheet decisions in your second colour.

## 9 · Now build the Pipeline
You have cleaned values, encoded categories, filled missing values, scaled numbers and split the rows.
In Sections 7–8, the learned steps were fitted on training rows and applied to test rows; now put them in an object that preserves their order.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

numeric_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median", add_indicator=ADD_INDICATOR)),
    ("scale", StandardScaler()),
])
nominal_steps = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ordinal_steps = OrdinalEncoder(categories=[ORDERS[c] for c in ORDERED_COLS],
                               handle_unknown="use_encoded_value", unknown_value=-1)
preprocessor = ColumnTransformer([
    ("num", numeric_steps, NUMERIC_COLS),
    ("nom", nominal_steps, NOMINAL_COLS),
    ("ord", ordinal_steps, ORDERED_COLS),
])
model_pipeline = Pipeline([("prep", preprocessor), ("model", LogisticRegression(max_iter=3000))])
print(model_pipeline)

### Task 15
Fit the pipeline below and confirm that its AUC matches Section 7 to three decimals.

My answer: ___

In [ ]:
model_pipeline.fit(train_frame, train_y)
pipeline_auc = roc_auc_score(test_y, model_pipeline.predict_proba(test_frame)[:, 1])
pipeline_test_matrix = model_pipeline.named_steps["prep"].transform(test_frame)
print(f"By hand: {manual_auc:.3f}   Pipeline: {pipeline_auc:.3f}")
print("Largest matrix difference:", float(np.max(np.abs(pipeline_test_matrix - test_matrix))))
record(9, "pipeline_holdout", f"indicator={ADD_INDICATOR}", pipeline_auc)

**You should see:** the same AUC and matrix values with the same choices.
A Pipeline preserves the result of these steps and reduces the chance of forgetting their order.
Cross-validation refits everything inside it on each training fold.

## 10 · Does the model matter?
Compare six model families on this same clinical table using identical folds and preparation.
These are fixed example settings, not a search for each family’s best configuration.

### Task 16
Which of these six models will win? Write down a prediction and a reason before running the comparison.

My answer: ___

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=3000),
    "KNeighbors": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(random_state=SEED),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=SEED),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=SEED),
    "MLP": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=600, random_state=SEED),
}
CV = StratifiedKFold(5, shuffle=True, random_state=SEED)
shared_folds = list(CV.split(ed, y))
race_rows = []
for name, estimator in models.items():
    candidate = Pipeline([("prep", clone(preprocessor)), ("model", clone(estimator))])
    scores = cross_val_score(candidate, ed, y, cv=shared_folds, scoring="roc_auc")
    race_rows.append({"model": name, "auc": scores.mean(), "fold_sd": scores.std()})
    record(10, "model_family", name, scores.mean())
race = pd.DataFrame(race_rows).sort_values("auc", ascending=False).reset_index(drop=True)
print(race.round(3))
fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
colors = ["#b66b46" if name == "MLP" else "#2f7e8e" for name in race["model"]]
bars = ax.barh(race["model"], race["auc"], color=colors)
ax.bar_label(bars, fmt="%.3f", padding=5)
ax.invert_yaxis()
ax.set(xlim=(0, 1.08), xlabel="Mean ROC-AUC across the same five folds", title="Six models on the clinical table")
plt.show()

Keep logistic regression unchanged and refit across 15 split seeds.
Compare the range with the earlier calculation by hand; it describes variability and is not a significance threshold.

In [ ]:
pipeline_seed_scores = []
for split_seed in range(15):
    a_rows, b_rows = train_test_split(np.arange(len(ed)), test_size=0.25,
                                      stratify=y, random_state=split_seed)
    fitted = clone(model_pipeline).fit(ed.iloc[a_rows], y.iloc[a_rows])
    auc = roc_auc_score(y.iloc[b_rows], fitted.predict_proba(ed.iloc[b_rows])[:, 1])
    pipeline_seed_scores.append(auc)
    record(10, "unchanged_split", f"seed={split_seed}", auc)
noise_range = np.ptp(pipeline_seed_scores)
top_three_range = np.ptp(race.head(3)["auc"])
all_models_range = np.ptp(race["auc"])
mlp_rank = int(race.index[race["model"].eq("MLP")][0]) + 1
print(f"Top-three spread: {top_three_range:.3f}   15-split range: {noise_range:.3f}   MLP rank: {mlp_rank}/6")
print("Largest difference from manual split scores:", float(np.max(np.abs(np.array(pipeline_seed_scores) - manual_seed_scores))))

**You should see:** the sorted model scores, the top-three spread, the split range and the neural network’s actual rank.

**Conclusion:** compare the measured model differences with the sensitivity to changing the split.
One neural network at these settings does not establish what every neural network could achieve.
Use the printed ranking to evaluate your prediction; a larger model is not automatically better.

## 11 · What actually moves the number
Selection can use held-out answers before a model ever sees the train/test split: this is data leakage.
First create coin-flip labels with no true predictive signal, then repeat the same mistake on the clinical table.

In [ ]:
rng = np.random.default_rng(7)
noise_features = rng.normal(size=(200, 500))
coin_labels = rng.integers(0, 2, 200)
selector = SelectKBest(f_classif, k=20).fit(noise_features, coin_labels)
coin_outside = cross_val_score(LogisticRegression(max_iter=2000), selector.transform(noise_features),
                               coin_labels, cv=5, scoring="roc_auc").mean()
coin_pipeline = Pipeline([("select", SelectKBest(f_classif, k=20)),
                          ("model", LogisticRegression(max_iter=2000))])
coin_inside = cross_val_score(coin_pipeline, noise_features, coin_labels, cv=5, scoring="roc_auc").mean()
print(f"Selection outside folds: {coin_outside:.3f}   Inside folds: {coin_inside:.3f}")
record(11, "coin_selection", "outside", coin_outside)
record(11, "coin_selection", "inside", coin_inside)

**You should see:** two scores from the same noise and labels.
Selecting columns against all labels lets the scoring answers influence the model inputs. Cross-validating only the classifier does not undo that.

Add the same fixed random columns to our clinical table and keep the selector size fixed.
Compare preparation and selection done on all rows with those steps refitted inside each training fold.

In [ ]:
noise_rng = np.random.default_rng(SEED)
noise_names = [f"random_{number}" for number in range(800)]
extra = pd.DataFrame(noise_rng.normal(size=(len(ed), 800)), columns=noise_names, index=ed.index)
wide = pd.concat([ed, extra], axis=1)
wide_preparation = ColumnTransformer([
    ("clinical", clone(preprocessor), MODEL_COLUMNS),
    ("random", "passthrough", noise_names),
])
wide_encoded = wide_preparation.fit_transform(wide, y)
wide_selector = SelectKBest(f_classif, k=30).fit(wide_encoded, y)
clinical_outside = cross_val_score(LogisticRegression(max_iter=3000), wide_selector.transform(wide_encoded),
                                  y, cv=shared_folds, scoring="roc_auc").mean()
inside_pipeline = Pipeline([("prepare", clone(wide_preparation)), ("select", SelectKBest(f_classif, k=30)),
                            ("model", LogisticRegression(max_iter=3000))])
clinical_inside = cross_val_score(inside_pipeline, wide, y, cv=shared_folds, scoring="roc_auc").mean()
preparation_gap = clinical_outside - clinical_inside
print(f"Outside: {clinical_outside:.3f}   Inside: {clinical_inside:.3f}   Gap: {preparation_gap:+.3f}")
record(11, "clinical_selection", "outside", clinical_outside)
record(11, "clinical_selection", "inside", clinical_inside)

Compare four measured changes, all in ROC-AUC units.
The last bar compares all six fixed model settings; it does not label any model family universally unsuitable.

In [ ]:
values = [top_three_range, noise_range, preparation_gap, all_models_range]
labels = ["Top three models", "Changing the split", "One preparation mistake", "All six models"]
fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")
bars = ax.bar(labels, values, color=["#2f7e8e", "#97bac1", "#b66b46", "#6d7984"], width=0.65)
ax.bar_label(bars, fmt="%.3f", padding=5)
ax.set(ylabel="Difference in ROC-AUC", title="What changed the score?",
       ylim=(min(0, min(values) * 1.2), max(values) * 1.25))
plt.show()
print(f"Model spread: {top_three_range:.3f}; preparation gap: {preparation_gap:.3f}.")

**Conclusion:** evaluation choices can change the score without improving predictions for new cases.
A high score does not reveal the mistake on its own; inspect what each step was allowed to learn.
Our first attempt filled values before splitting. Later examples fitted those operations on training rows only.

Save the measured results and use them to amend your Design Sheet.
State what you changed, why, and which evidence supports the revision.

In [ ]:
results.to_csv("results_s2.csv", index=False)
print(results.round({"auc": 3}))
print(f"Saved {len(results)} recorded results to results_s2.csv.")

Source: the supplied synthetic `ed_deterioration.csv`.
The follow-up measurements are generated from selected patients in that same table; the coin-flip example is a controlled simulation.
The original CSV is preserved. All comparison scores are calculated when you run the cells.
The example CSV is downloaded from [the course GitHub repository](https://raw.githubusercontent.com/nbrg-ppcu/appliedmedtech/session-02-ed-deterioration/notebooks/session_02/data/ed_deterioration.csv).
